# 05 — Drift Detection & MLOps

Looks at the 2025→2026 regulation era shift, computes drift metrics across feature distributions, and walks through the model promotion workflow.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path('../src')))
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# Load gold features for 2024/2025 (pre-2026)
import polars as pl
files_2024 = sorted(pathlib.Path('../data/silver/laps/season=2024').rglob('*.parquet'))
files_2025 = sorted(pathlib.Path('../data/silver/laps/season=2025').rglob('*.parquet'))
print(f'2024 files: {len(files_2024)}   2025 files: {len(files_2025)}')

In [ ]:
# Build features for each era
try:
    from pitwall.features.pace import build_pace_features, PACE_NUMERICAL
    era_2024 = build_pace_features(pl.concat([pl.read_parquet(f) for f in files_2024])) if files_2024 else None
    era_2025 = build_pace_features(pl.concat([pl.read_parquet(f) for f in files_2025])) if files_2025 else None
    for name, era in [('2024', era_2024), ('2025', era_2025)]:
        if era is not None:
            print(f'{name}: {len(era):,} rows')
except Exception as e:
    print(f'Feature build: {e}')

In [ ]:
# Drift metrics: KS test across features
def ks_drift(a, b):
    a = np.array(a.drop_nulls().to_numpy(), dtype=float)
    b = np.array(b.drop_nulls().to_numpy(), dtype=float)
    if len(a) < 10 or len(b) < 10:
        return None, None
    stat, pval = stats.ks_2samp(a, b)
    return stat, pval

def wasserstein(a, b):
    a = np.array(a.drop_nulls().to_numpy(), dtype=float)
    b = np.array(b.drop_nulls().to_numpy(), dtype=float)
    if len(a) < 10 or len(b) < 10:
        return None
    return stats.wasserstein_distance(a, b)

if era_2024 is not None and era_2025 is not None:
    print(f'{"Feature":<28} {"KS stat":>8} {"p-value":>10} {"Wasserstein":>12} {"Drift?":>8}')
    print('-' * 70)
    for feat in PACE_NUMERICAL:
        if feat in era_2024.columns and feat in era_2025.columns:
            ks, pval = ks_drift(era_2024[feat], era_2025[feat])
            w1 = wasserstein(era_2024[feat], era_2025[feat])
            if ks is not None:
                flag = 'DRIFT' if pval < 0.05 and ks > 0.1 else ''
                print(f'{feat:<28} {ks:>8.3f} {pval:>10.4f} {w1:>12.3f} {flag:>8}')

In [ ]:
# Visualise feature drift: lap_number distribution across seasons
if era_2024 is not None and era_2025 is not None:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.patch.set_facecolor('#080c14')
    fig.suptitle('Feature distributions: 2024 vs 2025', color='white', fontsize=13, fontweight='bold')
    feats_to_plot = ['tyre_age', 'stint_no', 'lap_number', 'rolling_std_5', 'stint_progress_ratio', 'race_progress']
    for ax, feat in zip(axes.flat, feats_to_plot):
        ax.set_facecolor('#0f172a')
        for sp in ax.spines.values(): sp.set_edgecolor('#1e293b')
        ax.tick_params(colors='#8b9bb4')
        if feat in era_2024.columns:
            a = era_2024[feat].drop_nulls().to_numpy()
            b = era_2025[feat].drop_nulls().to_numpy() if feat in era_2025.columns else np.array([])
            bins = np.linspace(min(a.min(), b.min() if len(b) else a.min()),
                               max(a.max(), b.max() if len(b) else a.max()), 35)
            ax.hist(a, bins=bins, alpha=0.65, color='#00d2be', density=True, label='2024')
            if len(b): ax.hist(b, bins=bins, alpha=0.65, color='#f59e0b', density=True, label='2025')
            ax.set_title(feat, color='white', fontsize=9)
            ax.legend(facecolor='#0f172a', edgecolor='#1e293b', labelcolor='white', fontsize=7)
    plt.tight_layout(); plt.show()

In [ ]:
# Promotion gate — what the weekly retrain checks
metrics = json.loads(pathlib.Path('../artifacts/candidate_smoke/metrics.json').read_text())
gate = {
    'mae < 2.0':            metrics['mae'] < 2.0,
    'coverage in [0.78,0.92]': 0.78 <= metrics.get('coverage_80_calibrated', metrics['coverage_80']) <= 0.92,
    'Hard MAE < 6.0':       metrics.get('per_compound', {}).get('HARD', 99) < 6.0,
}
promote = all(gate.values())
print('Promotion gate:')
for check, passed in gate.items():
    icon = '✓' if passed else '✗'
    print(f'  {icon}  {check}')
print()
print('Decision:', 'PROMOTE' if promote else 'HOLD — needs investigation')

In [ ]:
# MLflow experiment tracking (if server available)
try:
    import mlflow
    mlflow.set_tracking_uri('http://localhost:5000')
    runs = mlflow.search_runs(experiment_names=['pitwall-pace-prod'], max_results=5)
    print(runs[['run_id', 'status', 'metrics.mae', 'metrics.coverage_80', 'tags.mlflow.runName']].to_string())
except Exception as e:
    print(f'MLflow not reachable ({e}) — metrics live in artifacts/candidate_smoke/metrics.json')

In [ ]:
# Alert thresholds from monitoring/alerts.yml
with open('../monitoring/alerts.yml') as f:
    content = f.read()
print(content[:2000])